# Lesson 4.1: State, nodes and edges, by hand

**Goal:** run a small triage flow as a map (state + nodes + edges) with a 15-line runner you write yourself, before LangGraph.

**What you will be able to do**
- Write nodes that take the whole state and return a *partial update*.
- Join updates with a `merge` rule: overwrite most keys, append `notes`.
- Write plain and conditional edges, and a runner with a step brake.
- Score the flow on all 60 Brightcart incidents and find the bug the map shows.

In [1]:
# Setup: find the course repo's nightwatch/ folder (relative to this notebook) and load .env if present.
import os, sys
from pathlib import Path

for up in ("../../nightwatch", "../../../nightwatch", "../nightwatch", "nightwatch"):
    if Path(up, "brightcart").is_dir():
        NW = Path(up)
        break
else:
    raise FileNotFoundError("Open this notebook from <lesson>/notebook inside the course repo: it needs the nightwatch/ folder.")
sys.path.insert(0, str(NW.resolve()))

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))      # walks up from here and loads the repo's .env
except ImportError:
    pass
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
print("nightwatch folder:", NW.as_posix(), "| GROQ_API_KEY:", "found" if HAS_KEY else "not set")
print("This notebook makes no model calls, so no key is needed.")

nightwatch folder: ../../../nightwatch | GROQ_API_KEY: found
This notebook makes no model calls, so no key is needed.


## 1. Recall: two graders (lesson 3.6)

In lesson 3.6 we graded v0 with **two** graders: one checks the hidden label, one checks the shop's state, and on some runs
they disagree. Same habit today: we score the flow on all 60 incidents, not on one lucky example.

In [2]:
from brightcart.incidents import generate
INCIDENTS = {x.id: x for x in generate()}
print(len(INCIDENTS), "incidents, e.g.", INCIDENTS["upstream_deploy-0"].alert, "->", INCIDENTS["upstream_deploy-0"].label)

60 incidents, e.g. checkout error rate above 5% -> {'cause_service': 'payments', 'cause': 'bad_deploy', 'fix': ['rollback_deploy', 'payments', None]}


## 2. Key words

| Word | Meaning here |
|---|---|
| **state** | one dict that every step reads and adds to |
| **node** | a function that does one job: `node(state) -> update` |
| **partial update** | a small dict with only the keys the node changed |
| **edge** | "after this node, go there" (a node name) |
| **conditional edge** | a function that reads the state and returns the next node's name |
| **runner** | the loop that calls the nodes and follows the edges |

## 3. The idea: a flow as a map

```
read_alert -> check_health --(unhealthy?)--> read_logs --(upstream and hops < 3?)--> follow_upstream
                   |                            |   ^                                     |
                   | (all green)                |   +-------------------------------------+   (a cycle)
                   v                            v
              false_alarm -> END            classify -> END
```

Nodes are stations, the state is the train's cargo. No model call anywhere: rules only.

**Hand trace of `upstream_deploy-0`** (alert: *checkout error rate above 5%*), one row per step:

| step | node | partial update (without notes) | next |
|---|---|---|---|
| 1 | read_alert | suspect = checkout | check_health |
| 2 | check_health | unhealthy = [checkout, payments] | non-empty, so read_logs |
| 3 | read_logs | evidence = last 2 of 4 bad lines (both "upstream payments returned 500"), hops = 1 | upstream and 1 < 3, so follow_upstream |
| 4 | follow_upstream | suspect = payments | read_logs (the cycle) |
| 5 | read_logs | evidence = ["KeyError: 'currency' in serializer (v1.3.5)"], hops = 2 | no upstream, so classify |
| 6 | classify | cause = bad_deploy | END |

Final suspect: **payments**, not the service the pager named. Two more traces: `false_alarm-0` goes
read_alert, check_health (empty list), false_alarm: 3 steps. `disk_full-2` goes read_alert, check_health, read_logs
("No space left on device"), classify = disk_full: 4 steps.

## 4. Write it yourself

### 4a. The six nodes
Each node reads the state and returns only what it changed. `world(state)` builds a fresh simulated shop for the incident.

In [3]:
from brightcart.incidents import fresh
from brightcart.sim import World

SIGNS = [("401 Unauthorized", "expired_secret"), ("OOMKilled", "memory_leak"), ("No space left", "disk_full"),
         ("guarded by flag", "bad_flag"), ("db connection", "bad_config"), ("shedding load", "traffic_spike"),
         ("Error:", "bad_deploy")]

def world(state):
    return World(fresh(INCIDENTS[state["incident"]]))

def read_alert(state):
    first = state["alert"].split()[0]
    return {"suspect": first, "notes": ["alert names " + first]}

def check_health(state):
    h = world(state).health()
    return {"unhealthy": h["unhealthy_services"], "notes": [f"unhealthy: {h['unhealthy_services']}"]}

def read_logs(state):
    lines = [x for x in world(state).logs(state["suspect"], 20)
             if ("ERROR" in x or "WARN" in x) and "GC pause" not in x]
    return {"evidence": lines[-2:], "hops": state["hops"] + 1,
            "notes": [f"read {state['suspect']} logs: {len(lines)} bad lines"]}

def follow_upstream(state):
    line = next(e for e in state["evidence"] if "upstream" in e).split()
    callee = line[line.index("upstream") + 1]          # "... upstream payments returned 500 ..."
    return {"suspect": callee, "notes": [f"errors come from upstream {callee}"]}

def classify(state):
    text = " ".join(state["evidence"])
    cause = next((c for sign, c in SIGNS if sign in text), "unknown")
    return {"cause": cause, "notes": [f"cause = {cause}"]}

def false_alarm(state):
    return {"cause": "false_alarm", "notes": ["all green: false alarm, change nothing"]}

NODES = {f.__name__: f for f in (read_alert, check_health, read_logs, follow_upstream, classify, false_alarm)}

def start_state(incident_id):
    inc = INCIDENTS[incident_id]
    return {"incident": incident_id, "alert": inc.alert, "suspect": None, "unhealthy": [], "evidence": [],
            "hops": 0, "cause": None, "notes": []}

print(list(NODES))
print(start_state("upstream_deploy-0"))

['read_alert', 'check_health', 'read_logs', 'follow_upstream', 'classify', 'false_alarm']
{'incident': 'upstream_deploy-0', 'alert': 'checkout error rate above 5%', 'suspect': None, 'unhealthy': [], 'evidence': [], 'hops': 0, 'cause': None, 'notes': []}


### 4b. One node by hand
The whole state goes in, a small dict comes out. The state itself is not touched.

In [4]:
s = start_state("upstream_deploy-0")
print(read_alert(s))
print(check_health(s))
print("state unchanged:", s["suspect"], s["notes"])

{'suspect': 'checkout', 'notes': ['alert names checkout']}
{'unhealthy': ['checkout', 'payments'], 'notes': ["unhealthy: ['checkout', 'payments']"]}
state unchanged: None []


### 4c. merge, edges and the runner

In [5]:
import json

def merge(state, update):
    """How an update joins the state: 'notes' is appended, every other key is overwritten."""
    new = dict(state)
    for k, v in update.items():
        new[k] = state[k] + v if k == "notes" else v
    return new

# merge by hand: unhealthy is overwritten, notes grows from one line to two
s1 = merge(start_state("upstream_deploy-0"), read_alert(start_state("upstream_deploy-0")))
s2 = merge(s1, {"unhealthy": ["checkout", "payments"], "notes": ["unhealthy: ['checkout', 'payments']"]})
print(s2["unhealthy"], s2["notes"])

['checkout', 'payments'] ['alert names checkout', "unhealthy: ['checkout', 'payments']"]


In [6]:
def after_health(state):                      # conditional edge 1
    return "read_logs" if state["unhealthy"] else "false_alarm"

def after_logs(state):                        # conditional edge 2, with the cycle's brake
    upstream = any("upstream" in e for e in state["evidence"])
    return "follow_upstream" if upstream and state["hops"] < 3 else "classify"

END = "END"
EDGES = {"read_alert": "check_health",        # plain edge: always the same next node
         "check_health": after_health,        # conditional edge: a function picks the next node
         "read_logs": after_logs,
         "follow_upstream": "read_logs",      # a cycle: back to read the callee's logs
         "classify": END,
         "false_alarm": END}

def run(state, entry="read_alert", max_steps=10, show=True):
    node = entry
    for step in range(1, max_steps + 1):
        update = NODES[node](state)
        state = merge(state, update)
        if show:
            print(f"step {step} {node:16} update {json.dumps({k: v for k, v in update.items() if k != 'notes'})}")
        nxt = EDGES[node]
        node = nxt(state) if callable(nxt) else nxt
        if node == END:
            return state
    raise RuntimeError("step limit")

### 4d. Run it on the three incidents from the hand trace

In [7]:
for iid in ["upstream_deploy-0", "false_alarm-0", "disk_full-2"]:
    print("==", iid, "==")
    final = run(start_state(iid))
    print("suspect", final["suspect"], "| cause", final["cause"])
    print("notes  ", final["notes"], "\n")

== upstream_deploy-0 ==
step 1 read_alert       update {"suspect": "checkout"}
step 2 check_health     update {"unhealthy": ["checkout", "payments"]}
step 3 read_logs        update {"evidence": ["04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge", "04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge"], "hops": 1}
step 4 follow_upstream  update {"suspect": "payments"}
step 5 read_logs        update {"evidence": ["04:13 ERROR payments KeyError: 'currency' in serializer (v1.3.5)"], "hops": 2}
step 6 classify         update {"cause": "bad_deploy"}
suspect payments | cause bad_deploy
notes   ['alert names checkout', "unhealthy: ['checkout', 'payments']", 'read checkout logs: 4 bad lines', 'errors come from upstream payments', 'read payments logs: 1 bad lines', 'cause = bad_deploy'] 

== false_alarm-0 ==
step 1 read_alert       update {"suspect": "web"}
step 2 check_health     update {"unhealthy": []}
step 3 false_alarm      update 

Same order as the hand trace: 6 steps, 3 steps, 4 steps. The notes kept one line from every step.

## 5. Score it on all 60 incidents
Rules always give the same answer, so one run per incident is enough.

In [8]:
from collections import Counter

def score(run_fn=None):
    run_fn = run_fn or (lambda s: run(s, show=False))
    ok_svc = ok_cause = 0
    miss = Counter()
    for iid, inc in INCIDENTS.items():
        f = run_fn(start_state(iid))
        ok_svc += f["suspect"] == inc.label["cause_service"]
        ok_cause += f["cause"] == inc.label["cause"]
        if f["cause"] != inc.label["cause"]:
            miss[f"{inc.template} -> {f['cause']}"] += 1
    return ok_svc, ok_cause, dict(miss)

ok_svc, ok_cause, miss = score()
print(f"right service {ok_svc}/60 | right cause {ok_cause}/60 | misses: {miss}")

right service 54/60 | right cause 54/60 | misses: {'stuck_consumer -> false_alarm': 6}


**54/60.** All six misses are `stuck_consumer`, called a false alarm. A stuck consumer is a queue backing up while no
service looks unhealthy. The health report *does* list it, under `queues_over_limit`, but `check_health` throws that key away:

In [9]:
print(World(fresh(INCIDENTS["stuck_consumer-0"])).health())

{'healthy': False, 'unhealthy_services': [], 'queues_over_limit': {'email_outbox': 7734}}


## 6. The library version: your pieces, LangGraph's names

| You wrote | LangGraph |
|---|---|
| the state dict | a typed state (`TypedDict`) |
| `NODES[name] = fn` | `graph.add_node(name, fn)` |
| string edge | `graph.add_edge(a, b)` |
| edge function | `graph.add_conditional_edges(a, fn)` |
| `merge` (append for `notes`) | a **reducer** per key, e.g. `Annotated[list, operator.add]` |
| `run(state)` | `graph.compile().invoke(state)` |

LangGraph adds a drawing, streaming, saved state and parallel branches. The six node functions do not change.
Lesson 4.2 runs them in `StateGraph` and compares the final states with yours. A quick check that langgraph is installed:

In [10]:
import langgraph, importlib.metadata as md_
print("langgraph", md_.version("langgraph"))
from langgraph.graph import StateGraph, START, END as LG_END
print("StateGraph, START, END imported:", START, LG_END)

langgraph 1.2.22


StateGraph, START, END imported: __start__ __end__


## 7. Where this shows up in agents

Chapter 3's agent loop is a map with **two nodes**: `model` and `tools`. After the model, a switch: a tool call means
`tools`, otherwise END. Tools lead back to the model: that is the cycle. The state is the message list.
Lesson 4.3 builds this graph. Here it is with our own runner and a fake model, so it needs no API:

In [11]:
def fake_model(state):            # pretends to be the LLM: asks for one tool, then answers
    if not any(m["role"] == "tool" for m in state["messages"]):
        return {"messages": [{"role": "assistant", "tool_call": "health"}]}
    return {"messages": [{"role": "assistant", "content": "checkout is unhealthy"}]}

def fake_tools(state):
    return {"messages": [{"role": "tool", "content": "unhealthy: ['checkout']"}]}

def after_model(state):
    return "tools" if "tool_call" in state["messages"][-1] else END

def run_two(state, max_steps=6):
    nodes, edges, node = {"model": fake_model, "tools": fake_tools}, {"model": after_model, "tools": "model"}, "model"
    for _ in range(max_steps):
        state = {"messages": state["messages"] + nodes[node](state)["messages"]}   # messages: append
        nxt = edges[node]
        node = nxt(state) if callable(nxt) else nxt
        if node == END:
            return state
    raise RuntimeError("step limit")

for m in run_two({"messages": [{"role": "user", "content": "is checkout ok?"}]})["messages"]:
    print(m)

{'role': 'user', 'content': 'is checkout ok?'}
{'role': 'assistant', 'tool_call': 'health'}
{'role': 'tool', 'content': "unhealthy: ['checkout']"}
{'role': 'assistant', 'content': 'checkout is unhealthy'}


## 8. Traps
Each trap runs and shows what goes wrong, without stopping the notebook.

**Trap 1: editing the state inside a node.** Return only what changed. A node that appends to `state['notes']` in place changes the caller's dict, and then merge appends the line a second time.

In [12]:
def read_alert_bad(state):
    state["notes"].append("alert names " + state["alert"].split()[0])   # edits the input!
    return {"suspect": state["alert"].split()[0], "notes": ["alert names " + state["alert"].split()[0]]}

s = start_state("upstream_deploy-0")
after = merge(s, read_alert_bad(s))
print("input state was changed:", s["notes"])
print("note now appears twice :", after["notes"])

input state was changed: ['alert names checkout']
note now appears twice : ['alert names checkout', 'alert names checkout']


**Trap 2: a node picking its own route.** If a node returns `{'next': 'classify'}`, the runner just stores it as data; the route still comes from `EDGES`. Keep routes in the edges, where you can see them on the map.

In [13]:
def check_health_routes(state):
    out = check_health(state)
    out["next"] = "classify"            # the node tries to decide the route
    return out

NODES["check_health"] = check_health_routes
final = run(start_state("false_alarm-0"), show=False)
NODES["check_health"] = check_health    # put the real node back
print("node said 'classify', but the edge sent us to:", final["cause"], "| stray key:", final["next"])

node said 'classify', but the edge sent us to: false_alarm | stray key: classify


**Trap 3: a cycle with no brake.** Count hops and cap the steps. `upstream_deploy-0` needs 6 steps; with `max_steps=4` the runner raises instead of returning a half-done state.

In [14]:
try:
    run(start_state("upstream_deploy-0"), max_steps=4, show=False)
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: step limit


**Trap 4: a log that gets overwritten.** With a merge that overwrites every key, the final notes hold only the last line.

In [15]:
def merge_overwrite(state, update):
    return {**state, **update}

s = start_state("upstream_deploy-0")
for name in ["read_alert", "check_health"]:
    s = merge_overwrite(s, NODES[name](s))
print("notes after two steps:", s["notes"])

notes after two steps: ["unhealthy: ['checkout', 'payments']"]


## 9. Exercises

Each exercise has a starter cell (`# YOUR CODE HERE`) and a check cell. The checks need no API.

### Exercise 1 (the video's exercise): don't throw the queues away

Write `check_health_q` that also returns `"queues"` (the names from `queues_over_limit`), `after_health_q` that sends a
stuck queue to a new node `stuck_consumer`, and that node: it sets `suspect` to the queue's consumer
(`CONSUMERS` below) and `cause` to `"stuck_consumer"`. Then score all 60 again.

In [16]:
CONSUMERS = {"email_outbox": "email-worker"}

def check_health_q(state):
    # YOUR CODE HERE
    return check_health(state)

def after_health_q(state):
    # YOUR CODE HERE
    return after_health(state)

def stuck_consumer(state):
    # YOUR CODE HERE
    return {}

In [17]:
def run_q(state):
    nodes = {**NODES, "check_health": check_health_q, "stuck_consumer": stuck_consumer}
    edges = {**EDGES, "check_health": after_health_q, "stuck_consumer": END}
    node = "read_alert"
    for _ in range(10):
        state = merge(state, nodes[node](state))
        nxt = edges[node]
        node = nxt(state) if callable(nxt) else nxt
        if node == END:
            return state
    raise RuntimeError("step limit")

try:
    svc, cause, miss = score(run_q)
    print(f"right service {svc}/60 | right cause {cause}/60 | misses: {miss}")
    assert cause == 60, f"right cause is {cause}/60, expected 60/60"
    assert svc == 60, f"right service is {svc}/60, expected 60/60 (set suspect to the consumer)"
    f = run_q(start_state("stuck_consumer-0"))
    assert any("queue" in n or "stuck" in n for n in f["notes"]), "add a note from your new node(s)"
    print("✓ Exercise 1 passed")
except AssertionError as e:
    print("✗ Exercise 1 not yet:", e)

right service 54/60 | right cause 54/60 | misses: {'stuck_consumer -> false_alarm': 6}


✗ Exercise 1 not yet: right cause is 54/60, expected 60/60


### Exercise 2: merge with a reducer per key

Write `merge_r(state, update, reducers)` where `reducers` maps a key to a function `(old, new) -> value`; keys
not in `reducers` are overwritten. This is exactly what LangGraph's reducers do.

In [18]:
import operator

def merge_r(state, update, reducers):
    # YOUR CODE HERE
    return dict(state)

In [19]:
try:
    s = {"notes": ["a", "b"], "hops": 2, "cause": None}
    out = merge_r(s, {"notes": ["c"], "hops": 1}, {"notes": operator.add})
    assert out["notes"] == ["a", "b", "c"], f"notes should be appended, got {out['notes']}"
    assert out["hops"] == 1, f"hops should be overwritten, got {out['hops']}"
    assert s["notes"] == ["a", "b"], "do not change the input state"
    out2 = merge_r(s, {"hops": 5}, {"hops": max})
    assert out2["hops"] == 5 and merge_r(s, {"hops": 1}, {"hops": max})["hops"] == 2, "use the reducer for hops"
    print("✓ Exercise 2 passed")
except AssertionError as e:
    print("✗ Exercise 2 not yet:", e)

✗ Exercise 2 not yet: notes should be appended, got ['a', 'b']


### Exercise 3: return the path

Write `run_path(state)` that returns the list of node names visited (in order), using `NODES`, `EDGES` and `merge`,
with the same 10-step brake.

In [20]:
def run_path(state, entry="read_alert", max_steps=10):
    path = []
    # YOUR CODE HERE
    return path

In [21]:
try:
    p = run_path(start_state("upstream_deploy-0"))
    assert p == ["read_alert", "check_health", "read_logs", "follow_upstream", "read_logs", "classify"], f"got {p}"
    assert run_path(start_state("false_alarm-0")) == ["read_alert", "check_health", "false_alarm"]
    assert len(run_path(start_state("disk_full-2"))) == 4
    print("✓ Exercise 3 passed")
except AssertionError as e:
    print("✗ Exercise 3 not yet:", e)

✗ Exercise 3 not yet: got []


### Exercise 4: how long are the paths?

Using `run_path`, build `lengths`: a `Counter` of path length over all 60 incidents, and `looped`: how many
incidents went through `follow_upstream`.

In [22]:
lengths = Counter()
looped = 0
# YOUR CODE HERE

In [23]:
try:
    assert sum(lengths.values()) == 60, f"count all 60 incidents, got {sum(lengths.values())}"
    assert dict(lengths) == {3: 12, 4: 39, 6: 9}, f"got {dict(lengths)}"
    assert looped == 9, f"looped should be 9, got {looped}"
    print(dict(sorted(lengths.items())), "| looped:", looped)
    print("✓ Exercise 4 passed")
except AssertionError as e:
    print("✗ Exercise 4 not yet:", e)

✗ Exercise 4 not yet: count all 60 incidents, got 0


## Solutions (try first!)

In [24]:
CONSUMERS = {"email_outbox": "email-worker"}

def check_health_q(state):
    h = world(state).health()
    q = sorted(h["queues_over_limit"])
    return {"unhealthy": h["unhealthy_services"], "queues": q,
            "notes": [f"unhealthy: {h['unhealthy_services']} | queues over limit: {q}"]}

def after_health_q(state):
    if state.get("queues"):
        return "stuck_consumer"
    return "read_logs" if state["unhealthy"] else "false_alarm"

def stuck_consumer(state):
    q = state["queues"][0]
    return {"suspect": CONSUMERS.get(q, q), "cause": "stuck_consumer",
            "notes": [f"queue {q} is backing up: its consumer is stuck"]}

In [25]:
import operator

def merge_r(state, update, reducers):
    new = dict(state)
    for k, v in update.items():
        new[k] = reducers[k](state[k], v) if k in reducers else v
    return new

In [26]:
def run_path(state, entry="read_alert", max_steps=10):
    path, node = [], entry
    for _ in range(max_steps):
        path.append(node)
        state = merge(state, NODES[node](state))
        nxt = EDGES[node]
        node = nxt(state) if callable(nxt) else nxt
        if node == END:
            return path
    raise RuntimeError("step limit")

In [27]:
lengths = Counter()
looped = 0
for iid in INCIDENTS:
    p = run_path(start_state(iid))
    lengths[len(p)] += 1
    looped += "follow_upstream" in p